# **More Beautiful Soup**

---
---

In this notebook we will explore how to collect data from the web. The 'BeautifulSoup' package is introduced as a way to "scrape" the data from the pages.

Before exploring the widgets, run the code in the cell below to load all the required libraries.

In [ ]:
from bs4 import BeautifulSoup
import pandas as pd
import numpy as np
import requests as r

## **Soup Widgets**

The following cells demonstrate some of the ways we can apply Beautiful Soup to collecting data. Each pair or group will be pick one of the widgets to study. They will use the `requests` library that we imported that earlier.

 ## Your Task
<div class="alert alert-success" role="alert"> 
 <span style="color:black"> 
For this task, it is best to work in a pair. Each team will pick one of the widgets below. Run the code and write a short summary of what the widget does. Copy and paste the widget below and then add code to the widget or modify the existing code in some way. Be prepared to share your work with the class.  
 </span>
</div>

**Put summary of widget here**

In [ ]:
# Put modified widget here




## Widget 1

In [ ]:
# Assign the page to URL.
URL = "https://stackoverflow.com/questions/59916601/how-to-display-whole-table-in-the-output-in-jupyter-notebook"

# Request the page HTML information.
page = r.get(URL)

# Make the parsed beautiful soup object.
soup = BeautifulSoup(page.text)

# Pull all the <code> tags from the page.
codelist = soup.find_all("code")

for code in codelist:
    print(code)

## Widget 2

In [ ]:
# Assign the page to URL.
URL = "https://data.coloradoan.com/fires/"

# Request the page HTML information.
page = r.get(URL)

# Assign the tables from the (str) response into a list of DataFrames.
tables = pd.read_html(page.content)

# Assign the first table and display it.
first_table = tables[0]

# Show the entire table.
pd.set_option('display.max_columns', None) 
pd.set_option('display.max_rows', None)

# Display the table.
first_table

## Widget 3

In [ ]:
# Assign the page to URL.
URL = "https://www.spendwithpennies.com/easy-quiche-recipe/"
headers = { "User-Agent": "Jupyter Hub / Python 3.x Data Science Class" }
# Request the page HTML information.
page = r.get(URL, headers=headers)
soup = BeautifulSoup(page.text, 'html.parser')
imgs = soup.find_all('img')


In [ ]:
from pathlib import Path
img_folder = Path('Scraped_Images/')
img_folder.mkdir(exist_ok=True)

# Set the counter to 0.
i = 0

# Loop over all the figure tags.
for image in soup.find_all("img"):
    # Advance the counter. This will allow us to make different names. 
    i += 1
    
    # Identify the a tag and slice on href. Assign this to location.
    img_url = image.get('src')
    alt = image.get('alt')
    if (img_url[0:4]=='http'):
        print("Image "+str(i)+":"+img_url, end=' ')
        print("Alt: "+str(alt))
        img_data = r.get(img_url,headers=headers).content
        with open('Scraped_Images/image'+str(i)+'.jpg', 'wb') as handler:
            handler.write(img_data)

## Widget 4

In [ ]:
def make_soup(address):
    ''' Open the address and return the soup'''
    
    URL = "https://en.wikipedia.org/" + address
    page = r.get(URL,headers=headers)

    fresh_soup = BeautifulSoup(page.content, 'lxml')
    
    return fresh_soup

In [ ]:
def the_search(j, soup, look_for):
    '''Search the page for a given string or advance to a random page to keep looking.
                
            Parameters:
                j (int): Starting counter value
                soup (beautifulsoup object): The parsed html page
                look_for (str): String to look for on each page

    '''

    # Start a counter. 
    i = 0
    
    # List to collect the hyperlinks on the page.
    hyperlinks = []

    # Loop over all the <a> tags.
    for link in soup.find_all("a"):
        
        # Determine in the href is a string (we will skip if False).
        if type(link.get("href")) == str: 
            
            # Determine if the href has the string "wiki" in it in the first position.
            # We do this to make sure we only follow internal links.
            if link.get("href").find("wiki") == 1:
                
                # Append the interal links to the list. 
                hyperlinks.append(link.get("href"))
    
    # Loop over all the hyperlinks.
    for hyper in hyperlinks:
        
        # Search the links for the string that we are looking for.
        if hyper.find(look_for) > 0:
            
            # Print that it was found and stop the loop.
            print(f"found {look_for} at {j}")
            break

        # Check to see that we are at the end of the list. 
        if i == len(hyperlinks)-1:
            
            # Set a random target value for the next page.
            jump_val = np.random.randint(len(hyperlinks))
            
            # Using the random value, get a new link.
            next_address = hyperlinks[jump_val]
            
            # Print the address we head to.
            print(next_address)
            
            # Increase the page visit counter. 
            j += 1

            # Call make_soup to start the search on next page. 
            soup = make_soup(next_address)
            
            # Call the_search to search the next page. 
            the_search(j,soup, look_for)

        # Increase the hyperlink counter.
        i += 1

In [ ]:
# Set the starting page for the walk.
soup = make_soup("wiki/Association_football")
# Run the search.
the_search(0, soup, "China")

## Widget 5

In [ ]:
num_pages = 10
base_url = "https://www.goodreads.com/quotes?page="
headers = { "User-Agent": "Jupyter Hub / Python 3.x Data Science Class" }

print("Please wait. Building quote database...", end='')
# Build database of quotes from website
quotes = []
for page in range(num_pages):
    url = base_url + str(page)
    response = r.get(url, headers=headers)
    if response.status_code==200:
        soup = BeautifulSoup(response.text, 'html.parser')
        divs = soup.find_all("div", class_="quoteText")
        for div in divs:
            quote = div.text.split("―")[0]
            quotes.append(quote)
print("Done.")

In [ ]:
# Define ANSI codes for emphasis in text
emph = '\033[1;30;42m' #'\033[1m'
end_emph = '\033[0m'

search = input("Enter a search term:").lower()
total_quotes = len(quotes)
found_quotes = []
for quote in quotes:
    if quote.lower().find(search) > 0:
        found_quotes.append(quote)

print(f"\nThe term '{search}' was found in a total of {len(found_quotes)} quotations, listed below.\n")
for quote in found_quotes:
    q_with_emph = quote.replace(search, emph + search + end_emph)
    print(q_with_emph.strip()+"\n")